> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 三元组库入门（Triplet Store）

本 notebook 介绍 **Semantica 三元组库**模块：用工业界标准的 RDF 三元组库存储和查询知识图谱数据。

### 先看主线

09 篇走过属性图世界（Neo4j、节点 + 边、Cypher），这一篇走 RDF 世界（三元组 + SPARQL）——同一张图，两套生态，各有擅长。06 篇说过"三元组是入库层的标准单元"：`TripletExtractor` 抽出的三元组，存进的就是这里。

五节两段：

**连与建（第 1–2 节）—— 开门备料**

1. 第 1 节 连接：选后端（Blazegraph/Jena/RDF4J）连上服务
2. 第 2 节 造数据：用 `Triplet` 类定义三元组——注意主语谓语都是 URI，这是 RDF 世界的规矩

**增删查（第 3–5 节）—— 日常 CRUD**

3. 第 3 节 增：单条或批量写入
4. 第 4 节 查：SPARQL 检索
5. 第 5 节 删：不需要的三元组移除

提醒：本篇与 09 篇一样需要运行中的外部服务才能真跑；代码对"服务不在"做了优雅降级（捕获后跳过）。

## 什么是三元组库？

三元组库（Triplet Store，也叫 RDF Store）是为存取三元组优化的数据库：**主语 - 谓语 - 宾语**（Subject - Predicate - Object）。
例如：`Alice`（主语）`knows`（谓语）`Bob`（宾语）。

Semantica 支持：
- **Blazegraph**（默认，高性能）
- **Apache Jena**（擅长推理）
- **RDF4J**（标准 Java 框架）

In [ ]:
import sys
import os

# Add the project root to the path so we can import the local version of semantica
sys.path.append(os.path.abspath('../../'))

# If running in Google Colab, uncomment the following line to install dependencies
# !pip install -q semantica

In [ ]:
# Import the TripletStore class
from semantica.triplet_store import TripletStore
from semantica.semantic_extract.triplet_extractor import Triplet

## 1. 连接存储

用三元组库得先有运行中的后端。本例假设本地跑着一个 Blazegraph 实例。

> **注：** 如果没有起服务，下面的代码展示的是*应该怎么*连。

In [ ]:
# Connect to a Blazegraph instance
# You can also use backend="jena" or backend="rdf4j"
store = TripletStore(
    backend="blazegraph",
    endpoint="http://localhost:9999/blazegraph"
)

# Check connection status
if hasattr(store._store_backend, 'connected') and store._store_backend.connected:
    print(f"Successfully connected to {store.backend_type} store at {store.endpoint}")
else:
    print(f"Warning: Could not connect to {store.backend_type} at {store.endpoint}")
    print("Operations requiring a live store connection will be skipped or fail.")

## 2. 创建三元组

用 `Triplet` 类定义数据。

In [ ]:
# Define a single triplet
triplet1 = Triplet(
    subject="http://example.org/Alice",
    predicate="http://xmlns.com/foaf/0.1/knows",
    object="http://example.org/Bob"
)

print(f"Created triplet: {triplet1.subject} -> {triplet1.predicate} -> {triplet1.object}")

## 3. 添加数据

可以逐条添加，也可以批量。

In [ ]:
from semantica.utils.exceptions import ProcessingError

try:
    # Add a single triplet
    store.add_triplet(triplet1)
    print("Added single triplet successfully.")

    # Create more triplets
    triplets = [
        Triplet(
            subject="http://example.org/Bob",
            predicate="http://xmlns.com/foaf/0.1/knows",
            object="http://example.org/Charlie"
        ),
        Triplet(
            subject="http://example.org/Charlie",
            predicate="http://xmlns.com/foaf/0.1/knows",
            object="http://example.org/David"
        )
    ]

    # Bulk add
    store.add_triplets(triplets)
    print("Added bulk triplets successfully.")

except ProcessingError as e:
    print(f"Operation skipped: {e}")
except Exception as e:
    print(f"An error occurred: {e}")

## 4. 查询数据（SPARQL）

用 SPARQL 查询从库里取数据——RDF 世界的 SQL。

In [ ]:
# Simple query to get all triplets (limited to 10)
query = """
SELECT ?s ?p ?o
WHERE {
  ?s ?p ?o
}
LIMIT 10
"""

try:
    results = store.execute_query(query)
    print("Query Results:", results)
except ProcessingError as e:
    print(f"Query skipped: {e}")
except Exception as e:
    print(f"An error occurred: {e}")

## 5. 删除数据

不再需要的三元组移除即可。

In [ ]:
store.delete_triplet(triplet1)
print("Deleted triplet1")

## 小结

### 回顾主线

RDF 世界走完一圈：**连与建**（选后端、URI 化的三元组）开门备料，**增删查**（批量写、SPARQL 查、按需删）日常运转。与 09 篇合看：属性图和三元组库是同一张图的两套生态——要 Cypher 和 LPG 的灵活性去 09，要 SPARQL 和推理的标准化来本篇。

## 下一步

更复杂的操作——批量加载优化、事务、高级 SPARQL——见 `cookbook/advanced` 里的 **Advanced Triplet Store** 指南。